<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎙️ ScenA Audio - Expressive Speech & Scene Generator</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Google Colab T4 GPU Edition - Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0;'>Zero-Shot Voice Cloning • Multi-Speaker Dialogue • Sound Effects & Ambience | Powered by ScenA (LTX-2 audio DiT) + Gemma-3-12B</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-T4%20GPU-4285F4?style=for-the-badge&logo=google-colab&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

### Quick Start
1. **Runtime → Change runtime type → T4 GPU**
2. Run **Step 1** (installs packages and downloads ~16 GB of models, about 3–5 min).
3. Run **Step 2**, then open the **Cloudflare** link (or the Gradio share link) printed below it.
4. Pick **Narration** to have one cloned voice read your text, or **Scene script** for multi-speaker scenes with sound effects.

### Features & Optimizations
| Feature | Details |
|---|---|
| 🗣️ **Long text, no gibberish** | ScenA is trained on clips ≤ 20 s. Text is sized to the clip automatically and long text is split at sentence ends, generated part by part and joined |
| ⚡ **Models stay loaded** | The DiT, text readout and audio VAE are built once. A new seed, voice, duration or step count starts diffusion immediately |
| 🧠 **Smart text encoding** | Gemma-3-12B (4-bit) loads only when the text changes, runs on the real tokens instead of 1024 padded ones, and the result is cached |
| 🎯 **Accurate text features** | The text path runs in bf16 with fp32 normalisation, since fp16 overflows on Gemma-3 activations and weakens prompt following |
| 🎤 **Up to 3 voices** | Reference encodings are cached per file; clips are trimmed to a max length to keep generation fast |
| 🌐 **Dual links** | Cloudflare quick tunnel (`*.trycloudflare.com`) + Gradio public share |

*Text encoder: the ungated pre-quantized mirror `unsloth/gemma-3-12b-it-bnb-4bit`, so no Hugging Face token is needed.*

In [ ]:
#@title 📦 Step 1: GPU Check, Install, Clone & Download Models
# Installs the few missing packages, fetches ScenA at a pinned commit, and downloads
# ScenA + Gemma-3-12B (4-bit) + cloudflared in parallel. Safe to re-run: finished downloads are skipped.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

import shutil, stat, subprocess, sys, time, urllib.request
from concurrent.futures import ThreadPoolExecutor

ROOT = "/content" if os.path.isdir("/content") else os.getcwd()
REPO_DIR = os.path.join(ROOT, "scena")
CKPT_DIR = os.path.join(ROOT, "checkpoints")
GEMMA_DIR = os.path.join(ROOT, "gemma-3-12b-it-bnb-4bit")
SCENA_COMMIT = "c9b384e3b20615099b27238f4a49c21f92b67638"  # version this notebook is tuned on (26 Jul 2026)
CLOUDFLARED = "/tmp/cloudflared"
t0 = time.time()

# ── 1. GPU / RAM check ──
import psutil, torch
if not torch.cuda.is_available():
    raise SystemExit("❌ No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then run this cell again.")
props = torch.cuda.get_device_properties(0)
print(f"✅ GPU: {props.name} · {props.total_memory / 1024**3:.1f} GB VRAM · SM {props.major}.{props.minor}")
print(f"✅ RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB · Disk free: {shutil.disk_usage(ROOT).free / 1024**3:.0f} GB")

# ── 2. Python packages (Colab already ships torch/torchaudio/scipy/numpy) ──
print("\n📦 Installing packages...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.56", "accelerate>=1.0", "bitsandbytes>=0.45", "gradio>=5.0",
    "huggingface_hub>=0.34", "hf_xet", "safetensors", "av", "soundfile", "einops",
], check=True)
print("✅ Packages ready")

# ── 3. ScenA source at a pinned commit (a fresh checkout also wipes old runtime patches) ──
print("\n📥 Fetching ScenA source...")
git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
def git(*args):
    subprocess.run(["git", "-C", REPO_DIR, *args], check=True, env=git_env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    os.makedirs(REPO_DIR)
    git("init", "-q")
    git("remote", "add", "origin", "https://github.com/finmickey/scena.git")
git("fetch", "-q", "--depth", "1", "origin", SCENA_COMMIT)
git("checkout", "-q", "-f", "FETCH_HEAD")
git("reset", "-q", "--hard")
git("clean", "-q", "-fd")
assert os.path.exists(os.path.join(REPO_DIR, "packages/ltx-pipelines/src/ltx_pipelines/t2aud_ref_cond.py")), "ScenA checkout is incomplete"
print(f"✅ ScenA @ {SCENA_COMMIT[:10]}")

# ── 4. Model weights + cloudflared, downloaded in parallel ──
from huggingface_hub import hf_hub_download, snapshot_download

def get_scena(filename):
    hf_hub_download("mifinkelson/scena", filename, local_dir=CKPT_DIR)
    return f"ScenA {filename}"

def get_gemma():
    # Ungated pre-quantized 4-bit mirror of google/gemma-3-12b-it (no HF token needed)
    snapshot_download("unsloth/gemma-3-12b-it-bnb-4bit", local_dir=GEMMA_DIR,
                      allow_patterns=["*.json", "*.safetensors", "tokenizer.model", "*.jinja"])
    return "Gemma-3-12B-it 4-bit"

def get_cloudflared():
    if not os.path.exists(CLOUDFLARED):
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CLOUDFLARED)
        os.chmod(CLOUDFLARED, os.stat(CLOUDFLARED).st_mode | stat.S_IEXEC)
    return "cloudflared"

print("\n⬇️ Downloading ScenA (8.5 GB) + Gemma-3-12B 4-bit (8.5 GB) in parallel...")
jobs = [lambda: get_scena("scena.safetensors"), lambda: get_scena("audio_vae.safetensors"), get_gemma, get_cloudflared]
failed = False
with ThreadPoolExecutor(max_workers=len(jobs)) as pool:
    for fut in [pool.submit(job) for job in jobs]:
        try:
            print(f"✅ {fut.result()}")
        except Exception as e:
            failed = True
            print(f"❌ Download failed: {e}")
if failed:
    raise SystemExit("❌ Some downloads failed. Run this cell again (finished files are skipped).")

print(f"\n🎉 Step 1 done in {(time.time() - t0) / 60:.1f} min. Run Step 2 to launch the studio.")

In [ ]:
#@title 🚀 Step 2: Load ScenA Engine & Launch Gradio Studio
# Models load on the first Generate click and then stay in memory. Re-rolling the seed or changing
# voices, duration or steps skips all model loading; Gemma only reloads when the script text changes.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, hashlib, math, random, re, stat, subprocess, sys, threading, time, traceback, urllib.request
from collections import OrderedDict

import gradio as gr
import numpy as np
import psutil
import soundfile as sf
import torch
import transformers
from packaging import version
from transformers import AutoTokenizer

ROOT = "/content" if os.path.isdir("/content") else os.getcwd()
REPO_DIR = os.path.join(ROOT, "scena")
CKPT_DIR = os.path.join(ROOT, "checkpoints")
GEMMA_DIR = os.path.join(ROOT, "gemma-3-12b-it-bnb-4bit")
OUT_DIR = os.path.join(ROOT, "outputs")
EXAMPLES_DIR = os.path.join(REPO_DIR, "examples", "references")
SCENA_CKPT = os.path.join(CKPT_DIR, "scena.safetensors")
AUDIO_VAE_CKPT = os.path.join(CKPT_DIR, "audio_vae.safetensors")
os.makedirs(OUT_DIR, exist_ok=True)

_missing = [p for p in (SCENA_CKPT, AUDIO_VAE_CKPT, os.path.join(GEMMA_DIR, "config.json"), REPO_DIR) if not os.path.exists(p)]
if _missing:
    raise SystemExit(f"❌ Missing {_missing}. Run Step 1 first.")
for _p in ("packages/ltx-core/src", "packages/ltx-pipelines/src"):
    _p = os.path.join(REPO_DIR, _p)
    if _p not in sys.path:
        sys.path.insert(0, _p)

from ltx_core.components.guiders import MultiModalGuiderParams, create_multimodal_guider_factory
from ltx_core.components.noisers import GaussianNoiser
from ltx_core.components.schedulers import LTX2Scheduler
from ltx_core.loader.single_gpu_model_builder import SingleGPUModelBuilder as Builder
from ltx_core.model.audio_vae import (
    AUDIO_VAE_DECODER_COMFY_KEYS_FILTER, AUDIO_VAE_ENCODER_COMFY_KEYS_FILTER, VOCODER_COMFY_KEYS_FILTER,
    AudioDecoderConfigurator, AudioEncoderConfigurator, VocoderConfigurator,
    decode_audio as vae_decode_audio,
)
from ltx_core.model.transformer import LTXV_AUDIO_ONLY_RENAMING_MAP, LTXAudioOnlyModelConfigurator, X0Model
from ltx_core.text_encoders.gemma import (
    SCENA_AUDIO_ONLY_EMBEDDINGS_PROCESSOR_KEY_OPS, ScenaAudioOnlyEmbeddingsProcessorConfigurator,
)
from ltx_core.text_encoders.gemma.feature_extractor import FeatureExtractorV1, _norm_and_concat_padded_batch
from ltx_pipelines.t2aud_ref_cond import _AUDIO_LATENTS_PER_SECOND, _PLACEHOLDER_RES, _encode_ref_audio
from ltx_pipelines.utils.blocks import DiffusionStage
from ltx_pipelines.utils.denoisers import FactoryGuidedDenoiser, RefCondDenoiser
from ltx_pipelines.utils.media_io import decode_audio_from_file
from ltx_pipelines.utils.types import ModalitySpec

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
# T4 (SM 7.5) has no fast bf16, so the DiT runs in fp16 there; newer GPUs keep the native bf16.
DIT_DTYPE = torch.float16 if torch.cuda.get_device_capability(0)[0] < 8 else torch.bfloat16


def _bf16_matmul_ok():
    try:
        a = torch.randn(64, 64, device=DEVICE, dtype=torch.bfloat16)
        return bool(torch.isfinite((a @ a).float()).all())
    except Exception:
        return False


# Gemma-3 activations overflow fp16 (max 65504), which corrupts the text features ScenA listens to
# and weakens how well it follows the script. The text path therefore stays in bf16 like upstream;
# it runs only when the script changes, so its speed on T4 does not matter.
TEXT_DTYPE = torch.bfloat16 if _bf16_matmul_ok() else torch.float16
GEMMA_MAX_TOKENS = 1024  # upstream pads every prompt to 1024 tokens
NEW_TRANSFORMERS = version.parse(transformers.__version__) >= version.parse("4.56")
MODE_NARRATION = "🗣️ Narration (text read by Voice 1)"
MODE_SCENE = "🎬 Scene script (multi-speaker + sound effects)"
_dt = lambda d: str(d).replace("torch.", "")


def _feature_extractor_v1_forward(self, hidden_states, attention_mask, padding_side="left"):
    # Upstream normalises in the hidden-state dtype; the min/max/mean over ~10^6 Gemma values is done in fp32 here.
    encoded = torch.stack(hidden_states, dim=-1) if isinstance(hidden_states, (list, tuple)) else hidden_states
    normed = _norm_and_concat_padded_batch(encoded.float(), attention_mask)
    features = self.aggregate_embed(normed.to(self.aggregate_embed.weight.dtype))
    return (features, features) if self.is_av else (features, None)


FeatureExtractorV1.forward = _feature_extractor_v1_forward


def free_memory():
    gc.collect()
    torch.cuda.empty_cache()


class GenerationStopped(Exception):
    pass


STOP = threading.Event()


# ── Script helpers: prompt clean-up, speech-length estimate, long-text splitting ──
# ScenA was trained on scenes of at most 20 s. Asking it to fit more speech than the clip can hold
# is what makes it rush and slide into gibberish, so long text is split into parts that each fit.
MAX_REFS = 3            # the checkpoint has exactly 3 reference-voice slots
MAX_CLIP_S = 20.0       # longest clip the model was trained on (its RoPE positions stop at 20 s)
PART_SPEECH_S = 14.0    # speech budget per part when splitting long text (leaves room for breaths)
_CJK = "\u3040-\u30ff\u3400-\u4dbf\u4e00-\u9fff\uac00-\ud7af"
_QUOTE_FIX = str.maketrans({"“": '"', "”": '"', "„": '"', "«": '"', "»": '"'})


def count_words(text):
    text = re.sub(f"[{_CJK}]", " ", text)
    return sum(1 for w in text.split() if any(c.isalnum() for c in w))


def speech_seconds(text, wps):
    """Rough spoken length: words at the chosen pace, CJK characters at ~4.5 per second."""
    return count_words(text) / wps + len(re.findall(f"[{_CJK}]", text)) / 4.5


def quoted_speech_seconds(prompt, wps):
    return sum(speech_seconds(q, wps) for q in re.findall(r'"([^"]*)"', prompt))


def auto_duration(speech_s, extra_s=0.0):
    """Clip length that gives the speech room to breathe (too short = rushed, garbled speech)."""
    if speech_s <= 0:
        return 8.0
    return float(min(MAX_CLIP_S, max(3.0, round((speech_s * 1.05 + 1.0 + extra_s) * 2) / 2)))


def _group(pieces, budget_s, measure):
    """Greedily pack pieces into parts of balanced length, each within the budget."""
    total = sum(measure(p) for p in pieces)
    target = total / max(1, math.ceil(total / budget_s))
    parts, cur, cur_s = [], [], 0.0
    for p in pieces:
        s = measure(p)
        if cur and (cur_s + s > budget_s or cur_s >= target):
            parts.append(cur)
            cur, cur_s = [], 0.0
        cur.append(p)
        cur_s += s
    if cur:
        parts.append(cur)
    return parts


def split_text(text, budget_s, wps):
    """Split speech at sentence ends (then clauses, then words) into parts that fit the budget."""
    text = re.sub(r"\s+", " ", text).strip()
    if speech_seconds(text, wps) <= budget_s:
        return [text]
    pieces = []
    for sent in re.split(r"(?<=[.!?…])\s+|(?<=[。！？])", text):
        if not sent.strip():
            continue
        if speech_seconds(sent, wps) <= budget_s:
            pieces.append(sent.strip())
            continue
        for clause in re.split(r"(?<=[,;:-–])\s+", sent):
            if speech_seconds(clause, wps) <= budget_s:
                pieces.append(clause.strip())
            else:
                words, step = clause.split(), max(4, int(budget_s * wps))
                pieces += [" ".join(words[i:i + step]) for i in range(0, len(words), step)]
    measure = lambda p: speech_seconds(p, wps)
    return [" ".join(group) for group in _group(pieces, budget_s, measure)]


def narration_prompt(text, style="", scene=""):
    text = text.replace('"', "'").strip()
    says = f"says {style.strip()}" if style.strip() else "says"
    head = f"{scene.strip().rstrip('.')}. " if scene.strip() else ""
    return f'{head}The speaker from reference 1 {says}: "{text}"'


def clean_scene_prompt(prompt):
    """Fix the usual prompt mistakes: curly quotes, 'ref 1' shorthand, unquoted speech."""
    p = prompt.translate(_QUOTE_FIX)
    p = re.sub(r"\bref(?:erence)?[\s_.#-]*([1-3])\b", r"reference \1", p, flags=re.I)
    p = re.sub(r"^\s*speaker from", "The speaker from", p, flags=re.I)
    if '"' not in p:
        # 'The speaker from reference 1 says: Hello there' -> quote the spoken part
        m = re.search(r"reference [1-3]\b[^:\n]{0,80}:\s*", p)
        if m and p[m.end():].strip():
            p = p[:m.end()] + '"' + p[m.end():].strip() + '"'
    return re.sub(r"\s+", " ", p).strip()


def split_scene(prompt, budget_s, wps):
    """Split a quoted multi-speaker script between lines; over-long lines are split at sentences."""
    turns, prose = [], ""
    for piece in re.split(r'("[^"]*")', prompt):
        if len(piece) >= 2 and piece[0] == piece[-1] == '"':
            turns.append((prose.strip(), piece[1:-1].strip()))
            prose = ""
        else:
            prose += piece
    tail = prose.strip()
    if not turns or quoted_speech_seconds(prompt, wps) <= budget_s:
        return [prompt]
    units = []
    for lead, quote in turns:
        # continuation pieces reuse only the speaker attribution ("The speaker from reference 1 says:")
        attribution = re.split(r"(?<=[.!?])\s+", lead)[-1] if lead else ""
        for i, piece in enumerate(split_text(quote, budget_s, wps)):
            units.append((lead if i == 0 else attribution, piece))
    measure = lambda u: speech_seconds(u[1], wps)
    parts = [" ".join(f'{lead} "{q}"'.strip() for lead, q in group) for group in _group(units, budget_s, measure)]
    if tail:
        parts[-1] += " " + tail
    return parts


def plan_parts(mode, text, style, scene, wps, split_long):
    """Return [(prompt, estimated_speech_seconds), ...] - one entry per generated clip."""
    if mode == MODE_NARRATION:
        body = re.sub(r"\s+", " ", text.translate(_QUOTE_FIX)).strip()
        chunks = split_text(body, PART_SPEECH_S, wps) if split_long else [body]
        return [(narration_prompt(c, style, scene), speech_seconds(c, wps)) for c in chunks]
    prompt = clean_scene_prompt(text)
    prompts = split_scene(prompt, PART_SPEECH_S, wps) if split_long else [prompt]
    return [(p, quoted_speech_seconds(p, wps)) for p in prompts]


# ── ScenA engine: every model is built once and kept; Gemma is loaded only to encode new text ──
class ScenaEngine:
    GEMMA_VRAM = 9.0 * 1024**3  # 4-bit Gemma-3-12B weights (7.8 GB) + activations

    def __init__(self):
        self.dit = None
        self.dit_on_gpu = False
        self.proc = self.vae_enc = self.vae_dec = self.vocoder = self.tokenizer = None
        self.text_cache = OrderedDict()  # prompt -> text context for the DiT (on GPU, ~4 MB each)
        self.ref_cache = OrderedDict()   # (file hash, max seconds) -> (encoded reference, seconds)
        # Only used for its denoising loop; the transformer is passed in already built.
        self.stage = DiffusionStage(
            checkpoint_path=SCENA_CKPT, dtype=DIT_DTYPE, device=DEVICE,
            model_configurator=LTXAudioOnlyModelConfigurator, model_sd_ops=LTXV_AUDIO_ONLY_RENAMING_MAP,
        )
        self.scheduler = LTX2Scheduler()

    @staticmethod
    def _build(path, configurator, sd_ops, dtype):
        # Load the stored bf16 weights straight to the GPU and cast in place. The stock builder casts
        # through a second full copy, which briefly doubles VRAM (a 13 GB peak for the DiT).
        model = Builder(model_path=path, model_class_configurator=configurator, model_sd_ops=sd_ops).build(device=DEVICE)
        return model.to(dtype).eval()

    def ensure_small_models(self):
        if self.proc is not None:
            return
        print("⏳ Loading text readout (0.8B) + audio VAE & vocoder (0.2B)...")
        self.proc = self._build(SCENA_CKPT, ScenaAudioOnlyEmbeddingsProcessorConfigurator,
                                SCENA_AUDIO_ONLY_EMBEDDINGS_PROCESSOR_KEY_OPS, TEXT_DTYPE)
        # The VAE and vocoder are tiny, so they run in fp32 for the cleanest waveform.
        self.vae_enc = self._build(AUDIO_VAE_CKPT, AudioEncoderConfigurator, AUDIO_VAE_ENCODER_COMFY_KEYS_FILTER, torch.float32)
        self.vae_dec = self._build(AUDIO_VAE_CKPT, AudioDecoderConfigurator, AUDIO_VAE_DECODER_COMFY_KEYS_FILTER, torch.float32)
        self.vocoder = self._build(AUDIO_VAE_CKPT, VocoderConfigurator, VOCODER_COMFY_KEYS_FILTER, torch.float32)
        self.tokenizer = AutoTokenizer.from_pretrained(GEMMA_DIR)

    # ---- DiT residency ----
    def _dit_to_gpu(self, status):
        if self.dit is None:
            status("Loading ScenA DiT (3.3B) to GPU...")
            self.dit = X0Model(self._build(SCENA_CKPT, LTXAudioOnlyModelConfigurator,
                                           LTXV_AUDIO_ONLY_RENAMING_MAP, DIT_DTYPE)).eval()
        elif not self.dit_on_gpu:
            status("Moving parked DiT blocks back to GPU...")
            self.dit.to(DEVICE)
        self.dit_on_gpu = True

    def _make_room_for_gemma(self):
        """On a 15 GB T4 the DiT (6.6 GB) and Gemma (7.8 GB) don't fit together. Park just enough DiT blocks
        in CPU RAM (a few seconds each way) instead of reloading the DiT from disk after every text change."""
        free_memory()
        need = self.GEMMA_VRAM - torch.cuda.mem_get_info()[0]
        if not self.dit_on_gpu or need <= 0:
            return
        blocks = list(self.dit.velocity_model.transformer_blocks)
        block_bytes = sum(p.numel() * p.element_size() for p in blocks[0].parameters())
        n_park = min(len(blocks), math.ceil(need / block_bytes))
        if psutil.virtual_memory().available > n_park * block_bytes + 3 * 1024**3:
            for blk in blocks[-n_park:]:
                blk.to("cpu")
        else:
            print("⚠️ Low CPU RAM: freeing the DiT, it reloads from disk after text encoding")
            self.dit = None
        self.dit_on_gpu = False
        free_memory()

    # ---- text ----
    def _load_gemma(self):
        from transformers import BitsAndBytesConfig, Gemma3ForConditionalGeneration
        kwargs = {"device_map": {"": 0}, ("dtype" if NEW_TRANSFORMERS else "torch_dtype"): TEXT_DTYPE}
        if TEXT_DTYPE != torch.bfloat16:  # the checkpoint's own config computes in bf16
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=TEXT_DTYPE)
        return Gemma3ForConditionalGeneration.from_pretrained(GEMMA_DIR, **kwargs).eval()

    def _readout(self, hs):
        # Gemma ran on the real tokens only. Upstream left-pads to 1024 tokens, but the pads are masked
        # and RoPE is relative, so the hidden states are the same at ~10x less work. Rebuild that layout.
        n = hs.shape[1]
        full = hs.new_zeros((1, GEMMA_MAX_TOKENS, *hs.shape[2:]))
        full[:, -n:] = hs
        mask = torch.zeros((1, GEMMA_MAX_TOKENS), dtype=torch.long, device=DEVICE)
        mask[:, -n:] = 1
        ctx = self.proc.process_hidden_states(full, mask).video_encoding  # ScenA reads the "video" slot
        if not torch.isfinite(ctx).all():
            raise RuntimeError("Text encoding produced NaN/inf values.")
        return ctx.to(DIT_DTYPE)

    @torch.inference_mode()
    def encode_texts(self, prompts, status):
        """Return one DiT text context per prompt; only prompts not seen before go through Gemma."""
        todo = [p for p in dict.fromkeys(prompts) if p not in self.text_cache]
        if todo:
            self._make_room_for_gemma()
            status(f"Loading Gemma-3-12B to encode {len(todo)} new text(s)...")
            gemma = self._load_gemma()
            hidden = {}
            try:
                for p in todo:
                    ids = self.tokenizer(p.strip(), truncation=True, max_length=GEMMA_MAX_TOKENS,
                                         return_tensors="pt").input_ids.to(DEVICE)
                    out = gemma.model(input_ids=ids, attention_mask=torch.ones_like(ids),
                                      output_hidden_states=True, use_cache=False)
                    hidden[p] = torch.stack(out.hidden_states, dim=-1)  # [1, tokens, 3840, 49 layers]
                    del out
            finally:
                del gemma
                free_memory()
            for p, hs in hidden.items():
                self.text_cache[p] = self._readout(hs)
            del hidden
            while len(self.text_cache) > 64:
                self.text_cache.popitem(last=False)
            free_memory()
        for p in prompts:
            self.text_cache.move_to_end(p)
        return [self.text_cache[p] for p in prompts]

    # ---- reference voices ----
    @torch.inference_mode()
    def encode_ref(self, path, max_s):
        with open(path, "rb") as f:
            key = (hashlib.md5(f.read()).hexdigest(), float(max_s))
        if key in self.ref_cache:
            self.ref_cache.move_to_end(key)
            return self.ref_cache[key]
        audio = decode_audio_from_file(path, DEVICE, max_duration=float(max_s))
        if audio is None:
            raise gr.Error(f"Could not read audio from {os.path.basename(path)}")
        wf = audio.waveform.float()
        if wf.shape[1] > 2:  # surround -> mono (the VAE takes mono or stereo)
            wf = wf.mean(dim=1, keepdim=True)
        seconds = wf.shape[-1] / audio.sampling_rate
        if seconds < 1.0:
            raise gr.Error(f"{os.path.basename(path)} is only {seconds:.1f} s; use at least 3 s of clean speech.")
        audio = type(audio)(waveform=wf, sampling_rate=audio.sampling_rate)
        self.ref_cache[key] = (_encode_ref_audio(audio, self.vae_enc, DEVICE, DIT_DTYPE), seconds)
        while len(self.ref_cache) > 16:
            self.ref_cache.popitem(last=False)
        return self.ref_cache[key]

    # ---- diffusion + decode ----
    @torch.inference_mode()
    def generate_part(self, ctx_p, ctx_n, refs, duration, steps, cfg, seed, on_step, status):
        self._dit_to_gpu(status)
        guider = create_multimodal_guider_factory(params=MultiModalGuiderParams(cfg_scale=float(cfg)), negative_context=ctx_n)
        denoiser = RefCondDenoiser(
            inner_denoiser=FactoryGuidedDenoiser(v_context=None, a_context=ctx_p, video_guider_factory=None,
                                                 audio_guider_factory=guider),
            ref_conds=refs, max_ref_conds=MAX_REFS,
        )

        def step(transformer, video_state, audio_state, sigmas, i):
            if STOP.is_set():
                raise GenerationStopped()
            out = denoiser(transformer, video_state, audio_state, sigmas, i)
            on_step(i + 1)
            return out

        sigmas = self.scheduler.execute(steps=int(steps)).to(dtype=torch.float32, device=DEVICE)
        noiser = GaussianNoiser(generator=torch.Generator(device=DEVICE).manual_seed(int(seed)))
        # max_batch_size=2 runs the CFG cond + uncond passes as one batch instead of two forwards.
        _, audio_state = self.stage.run(
            self.dit, step, sigmas, noiser, _PLACEHOLDER_RES, _PLACEHOLDER_RES,
            round(duration * _AUDIO_LATENTS_PER_SECOND), _AUDIO_LATENTS_PER_SECOND,
            video=None, audio=ModalitySpec(context=ctx_p), max_batch_size=2,
        )
        latent = audio_state.latent
        if not torch.isfinite(latent).all():
            raise RuntimeError("The DiT produced NaN/inf values. Try another seed or a lower guidance scale.")
        return latent

    @torch.inference_mode()
    def decode(self, latent):
        audio = vae_decode_audio(latent.float(), self.vae_dec, self.vocoder)
        wav = audio.waveform.float().cpu().numpy()
        return (wav[0] if wav.ndim == 3 else wav), audio.sampling_rate  # (channels, samples)

    def release(self):
        for name in ("dit", "proc", "vae_enc", "vae_dec", "vocoder"):
            setattr(self, name, None)
        self.text_cache.clear()
        self.ref_cache.clear()
        free_memory()


# ── Output post-processing ──
def trim_silence(wav, sr, floor_db=-40.0):
    """Trim quiet edges of a (channels, samples) clip, keeping a short natural margin."""
    hop = max(1, int(sr * 0.02))
    frames = wav.shape[1] // hop
    if frames < 3:
        return wav
    env = np.abs(wav[:, :frames * hop]).max(axis=0).reshape(frames, hop).max(axis=1)
    loud = np.nonzero(env > env.max() * 10 ** (floor_db / 20))[0]
    if len(loud) == 0:
        return wav
    start = max(0, loud[0] * hop - int(0.08 * sr))
    end = min(wav.shape[1], (loud[-1] + 1) * hop + int(0.15 * sr))
    return wav[:, start:end]


def join_parts(wavs, sr, gap_s=0.3):
    """Join generated parts: trim their silent edges, fade in/out 10 ms, insert a short natural pause."""
    fade = int(0.01 * sr)
    ramp = np.linspace(0.0, 1.0, fade, dtype=np.float32)
    out = []
    for i, w in enumerate(wavs):
        w = trim_silence(w, sr).astype(np.float32, copy=True)
        if w.shape[1] > 2 * fade:
            w[:, :fade] *= ramp
            w[:, -fade:] *= ramp[::-1]
        out.append(w)
        if i < len(wavs) - 1:
            out.append(np.zeros((w.shape[0], int(gap_s * sr)), dtype=np.float32))
    return np.concatenate(out, axis=1)


def normalize_peak(wav, peak=0.98):
    m = float(np.abs(wav).max()) if wav.size else 0.0
    return wav * (peak / m) if m > peak else wav


_old_engine = globals().pop("engine", None)  # re-running this cell: release the previous engine's VRAM
if hasattr(_old_engine, "release"):
    _old_engine.release()
globals().pop("pipe", None)  # pipeline object from the previous version of this notebook
del _old_engine
free_memory()
engine = ScenaEngine()


# ── Generation handler ──
def generate(mode, text, style, scene, ref1, ref2, ref3, auto_dur, duration, split_long, pace, steps, cfg, seed,
             ref_max_s, progress=gr.Progress()):
    STOP.clear()
    t_start = time.time()
    text = (text or "").strip()
    if not text:
        raise gr.Error("Please enter a script.")
    voices = [ref1] if mode == MODE_NARRATION else [ref1, ref2, ref3]
    if not any(voices):
        raise gr.Error("Upload at least one voice reference (Voice 1).")

    parts = plan_parts(mode, text, style or "", scene or "", float(pace), bool(split_long))
    extra = 1.0 if mode == MODE_SCENE else 0.0
    durations = [auto_duration(s, extra) if auto_dur else float(duration) for _, s in parts]
    notes = []
    for k, ((_, s), d) in enumerate(zip(parts, durations)):
        if s > d:
            where = f"part {k + 1}" if len(parts) > 1 else "the script"
            notes.append(f"⚠️ {where} needs ≈{s:.0f} s of speech but the clip is {d:g} s: expect rushed or garbled "
                         "speech (turn on Auto duration / long-text splitting)")
    used = sorted({int(n) for p, _ in parts for n in re.findall(r"reference ([1-3])\b", p)})
    for n in used:
        if n > len(voices) or not voices[n - 1]:
            notes.append(f"⚠️ the script mentions reference {n} but Voice {n} is empty")
    seed_val = random.randint(0, 2**31 - 1) if int(seed) < 0 else int(seed)
    total_steps = len(parts) * int(steps)

    def status(msg):
        print(f"  {msg}")
        progress(None, desc=msg)

    try:
        progress(0, desc="Preparing...")
        engine.ensure_small_models()

        t = time.time()
        refs, ref_desc = {}, []
        for i, path in enumerate(voices):
            if path:
                refs[i], secs = engine.encode_ref(path, ref_max_s)
                ref_desc.append(f"Voice {i + 1}: {secs:.1f} s")
        t_ref = time.time() - t

        t = time.time()
        n_new = len(({p for p, _ in parts} | {""}) - set(engine.text_cache))
        ctxs = engine.encode_texts([p for p, _ in parts] + [""], status)  # "" = unconditional context for CFG
        t_text = time.time() - t

        t = time.time()
        wavs, sr = [], None
        for k, ((prompt, _), dur) in enumerate(zip(parts, durations)):
            label = f"Part {k + 1}/{len(parts)} · " if len(parts) > 1 else ""
            def on_step(i, k=k, label=label):
                progress((k * int(steps) + i) / total_steps, desc=f"{label}step {i}/{int(steps)}")
            print(f"🎙️ {label}{dur:g} s clip · seed {seed_val}\n   {prompt}")
            latent = engine.generate_part(ctxs[k], ctxs[-1], refs, dur, steps, cfg, seed_val, on_step, status)
            wav, sr = engine.decode(latent)
            wavs.append(wav)
        t_gen = time.time() - t

        wav = normalize_peak(wavs[0] if len(wavs) == 1 else join_parts(wavs, sr))
        out_path = os.path.join(OUT_DIR, f"scena_{time.strftime('%Y%m%d_%H%M%S')}_seed{seed_val}.wav")
        sf.write(out_path, wav.T, sr, subtype="PCM_16")
    except GenerationStopped:
        free_memory()
        return None, "🛑 Stopped."
    except torch.cuda.OutOfMemoryError:
        free_memory()
        return None, "❌ Out of GPU memory. Use shorter voice references or fewer voices, then try again."
    except gr.Error:
        raise
    except Exception as e:
        free_memory()
        print(traceback.format_exc())
        return None, f"❌ {type(e).__name__}: {e}\n\n{traceback.format_exc()[-1500:]}"

    total = time.time() - t_start
    lines = [
        f"✅ {wav.shape[1] / sr:.1f} s of audio in {total:.0f} s · seed {seed_val}",
        f"⏱️ voices {t_ref:.1f} s · text {t_text:.1f} s ({'Gemma loaded for ' + str(n_new) + ' new text(s)' if n_new else 'cached'})"
        f" · diffusion + decode {t_gen:.1f} s",
        f"🎤 {' · '.join(ref_desc)}",
        f"🧩 {len(parts)} part(s): " + ", ".join(f"{d:g} s" for d in durations),
        *notes,
        "",
        *[f"[{k + 1}] {p}" for k, (p, _) in enumerate(parts)],
    ]
    print(f"✅ Saved {out_path} ({total:.0f} s)")
    return out_path, "\n".join(lines)


def stop_generation():
    STOP.set()


def estimate(mode, text, style, scene, pace, split_long, auto_dur, duration):
    text = (text or "").strip()
    if not text:
        return "📝 Type a script to see how long it will be."
    parts = plan_parts(mode, text, style or "", scene or "", float(pace), bool(split_long))
    speech = sum(s for _, s in parts)
    head = f"📝 {count_words(text)} words · ≈ {speech:.0f} s of speech"
    if len(parts) > 1:
        return f"{head} → **{len(parts)} parts** of ≤ 20 s, generated one by one and joined into one file"
    d = auto_duration(speech, 1.0 if mode == MODE_SCENE else 0.0) if auto_dur else float(duration)
    if speech > d:
        return f"{head} → one {d:g} s clip ⚠️ **too much text for the clip** (turn on Auto duration / long-text splitting)"
    return f"{head} → one {d:g} s clip"


def switch_mode(mode):
    narration = mode == MODE_NARRATION
    return (
        gr.update(label="🗣️ Text to speak" if narration else "🎬 Scene script",
                  placeholder=NARRATION_PLACEHOLDER if narration else SCENE_PLACEHOLDER),
        gr.update(visible=narration),
        gr.update(visible=not narration),
        gr.update(visible=not narration),
    )


# ── Gradio UI (AIQUEST standard) ──
NARRATION_PLACEHOLDER = "Type what Voice 1 should say. Long text is split into ~15 s parts automatically."
# One-speaker preset showing the prompt structure: setting -> sound -> speaker + delivery -> "speech" -> sound -> ...
SINGLE_SPEAKER_PRESET = ('A cozy kitchen in the morning. Rain taps gently on the window and a kettle hums in the '
                         'background. The speaker from reference 1 says warmly: "Good morning! The coffee is almost '
                         'ready." A spoon clinks against a mug. The speaker from reference 1 laughs softly and says: '
                         '"Honestly, I could get used to rainy days like this."')
SCENE_PLACEHOLDER = SINGLE_SPEAKER_PRESET

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; border: none !important; }
.status-line { text-align: center; color: #6b7280; font-size: 13px; margin: -8px 0 14px 0; }
.footer { text-align: center; padding: 22px; margin-top: 32px; border-top: 1px solid #e5e7eb; color: #6b7280; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">🎙️ ScenA Audio - Expressive Speech & Scene Generator</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; Zero-shot voice cloning · multi-speaker dialogue · sound effects</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""

STATUS_HTML = (f'<div class="status-line">⚡ {GPU_NAME} · DiT {_dt(DIT_DTYPE)} · text encoder {_dt(TEXT_DTYPE)} · '
               'models stay loaded between runs · Gemma reloads only when the text changes</div>')

FOOTER_HTML = """
<div class="footer">
  <p style="font-size: 15px; margin: 4px 0;">🎙️ Created by <strong>AIQUEST Academy</strong></p>
  <p style="font-size: 13px; margin: 8px 0;">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">YouTube</a> |
    <a href="https://x.com/aiquestacademy" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">X (Twitter)</a> |
    <a href="https://aiquest.site" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px;">aiquest.site</a>
  </p>
  <p style="font-size: 12px; margin: 4px 0;">⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved</p>
</div>
"""

TIPS_MD = """
### 💡 Tips
* **Narration**: just type the words. Voice 1 is cloned; long text is split at sentence ends into ≤ 20 s parts and joined.
* **Scene script**: refer to speakers as **"the speaker from reference 1 / 2 / 3"**, put spoken words in **"double quotes"**, and describe sounds in plain prose (*rain drums on a tin roof*).
* **Scene structure** (see the first example, 1 speaker):
  `Setting.` → `Background sounds.` → `The speaker from reference 1 says warmly:` → `"Spoken words."` → `A sound effect.` → `The speaker from reference 1 laughs and says:` → `"More words."`
* **Voices**: 6–15 s of clean, single-speaker speech works best. Longer clips are cut to *Max reference length*.
* **Rushed or garbled speech** = too much text for the clip: keep Auto duration on, or lower *Speaking pace*. Long pauses = raise it.
* **Speed**: the first run loads everything (~2–3 min). After that a new seed/voice/duration starts straight away; new text reloads Gemma (~1 min on T4).
"""

GR_MAJOR = int(gr.__version__.split(".")[0])
blocks_kwargs = {} if GR_MAJOR >= 6 else {"theme": gr.themes.Soft(), "css": CSS}

with gr.Blocks(title="ScenA Audio | AIQUEST", **blocks_kwargs) as demo:
    gr.HTML(BRAND_HTML)
    gr.HTML(STATUS_HTML)

    with gr.Row():
        with gr.Column(scale=1):
            mode_in = gr.Radio([MODE_NARRATION, MODE_SCENE], value=MODE_NARRATION, label="Mode")
            text_in = gr.Textbox(label="🗣️ Text to speak", placeholder=NARRATION_PLACEHOLDER, lines=7)
            with gr.Row(visible=True) as narration_row:
                style_in = gr.Textbox(label="Delivery (optional)", placeholder="warmly · with excitement · in a whisper")
                scene_in = gr.Textbox(label="Setting (optional)", placeholder="In a quiet recording studio")
            estimate_md = gr.Markdown(estimate(MODE_NARRATION, "", "", "", 2.8, True, True, 10.0))
            ref1_in = gr.Audio(label="🎤 Voice 1 (reference 1)", type="filepath")
            ref2_in = gr.Audio(label="🎤 Voice 2 (reference 2)", type="filepath", visible=False)
            ref3_in = gr.Audio(label="🎤 Voice 3 (reference 3)", type="filepath", visible=False)

        with gr.Column(scale=1):
            with gr.Accordion("⚙️ Generation Settings", open=False):
                auto_in = gr.Checkbox(label="Auto duration (fit clip length to the text)", value=True)
                duration_in = gr.Slider(label="Clip duration (s) when Auto is off", minimum=2, maximum=20, value=10, step=0.5)
                split_in = gr.Checkbox(label="Split long text into ≤ 20 s parts", value=True)
                pace_in = gr.Slider(label="Speaking pace (words / second)", minimum=1.8, maximum=4.0, value=2.8, step=0.1)
                steps_in = gr.Slider(label="Inference steps (upstream default 60)", minimum=10, maximum=80, value=30, step=5)
                cfg_in = gr.Slider(label="Guidance scale", minimum=1.0, maximum=12.0, value=7.0, step=0.5)
                seed_in = gr.Number(label="Seed (-1 = random)", value=-1, precision=0)
                refmax_in = gr.Slider(label="Max reference length (s)", minimum=5, maximum=20, value=15, step=1)
            with gr.Row():
                gen_btn = gr.Button("🎵 Generate Audio", variant="primary", size="lg")
                stop_btn = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
            audio_out = gr.Audio(label="🔊 Generated Audio", type="filepath")
            status_out = gr.Textbox(label="Status", lines=9, interactive=False)
            gr.Markdown(TIPS_MD)

    ex1, ex2 = (os.path.join(EXAMPLES_DIR, f"reference_{i}.wav") for i in (1, 2))
    gr.Examples(
        examples=[
            [MODE_SCENE, SINGLE_SPEAKER_PRESET, "", "", ex1, None, None],
            [MODE_SCENE, 'The speaker from reference 1 says: "The taxi drivers are on strike again." The speaker from '
                         'reference 2 says: "What for?" The speaker from reference 1 says: "They want the government to '
                         'reduce the price of the gasoline." The speaker from reference 2 says: "It is really a hot potato."',
             "", "", ex1, ex2, None],
            [MODE_SCENE, 'A farm at sunrise: a rooster crows. Chickens cluck softly throughout. The speaker from reference 1 '
                         'says with a yawn: "Way too early for this." The speaker from reference 2 chuckles: "Welcome to '
                         'country life." The rooster crows again.', "", "", ex1, ex2, None],
            [MODE_NARRATION, "Welcome back to the channel! Today we are testing a brand new open-source audio model that "
                             "can clone any voice from a few seconds of speech. It can even build full scenes with several "
                             "speakers, background ambience and sound effects. Let's see if it lives up to the hype.",
             "with excitement", "", ex1, None, None],
        ],
        inputs=[mode_in, text_in, style_in, scene_in, ref1_in, ref2_in, ref3_in],
        label="Examples (bundled ScenA reference voices)",
        cache_examples=False,
    )
    gr.HTML(FOOTER_HTML)

    mode_in.change(switch_mode, inputs=mode_in, outputs=[text_in, narration_row, ref2_in, ref3_in], queue=False)
    est_inputs = [mode_in, text_in, style_in, scene_in, pace_in, split_in, auto_in, duration_in]
    for comp in est_inputs:
        comp.change(estimate, inputs=est_inputs, outputs=estimate_md, queue=False, show_progress="hidden")
    gen_btn.click(
        generate,
        inputs=[mode_in, text_in, style_in, scene_in, ref1_in, ref2_in, ref3_in, auto_in, duration_in, split_in,
                pace_in, steps_in, cfg_in, seed_in, refmax_in],
        outputs=[audio_out, status_out],
    )
    stop_btn.click(stop_generation, queue=False)


# ── Launch: Gradio share link + Cloudflare quick tunnel ──
SERVER_PORT = 7860


def start_cloudflare_tunnel(port):
    found = {"url": None}
    binary = "/tmp/cloudflared"
    try:
        if not os.path.exists(binary):
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", binary)
            os.chmod(binary, os.stat(binary).st_mode | stat.S_IEXEC)
        subprocess.run(["pkill", "-f", "cloudflared tunnel"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        tunnel = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    except Exception as e:
        print(f"⚠️ Cloudflare tunnel unavailable: {e}")
        return found

    def _watch():
        for line in tunnel.stdout:  # keep draining so cloudflared never blocks on a full pipe
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if m and not found["url"]:
                found["url"] = m.group(0)

    threading.Thread(target=_watch, daemon=True).start()
    return found


_prev_demo = globals().get("demo_prev")  # re-running this cell: free the port held by the previous UI
if _prev_demo is not None:
    try:
        _prev_demo.close()
    except Exception:
        pass
demo_prev = demo

print("🚀 Launching ScenA Studio...")
cf = start_cloudflare_tunnel(SERVER_PORT)
demo.queue(max_size=20, default_concurrency_limit=1)
launch_kwargs = dict(server_name="127.0.0.1", server_port=SERVER_PORT, share=True, inline=False, show_error=True,
                     prevent_thread_lock=True, quiet=True, allowed_paths=[OUT_DIR, EXAMPLES_DIR])
if GR_MAJOR >= 5:
    launch_kwargs["ssr_mode"] = False
if GR_MAJOR >= 6:
    launch_kwargs.update(theme=gr.themes.Soft(), css=CSS)
demo.launch(**launch_kwargs)

for _ in range(40):  # give the tunnel up to ~20 s to report its URL
    if cf["url"]:
        break
    time.sleep(0.5)
print("\n" + "=" * 65)
print("🎙️ ScenA Studio is LIVE!")
print("=" * 65)
print(f"🌐 Cloudflare tunnel: {cf['url'] or '(still connecting, check again in a moment)'}")
print(f"🔗 Gradio share:      {getattr(demo, 'share_url', None) or '(unavailable, use the Cloudflare link)'}")
print(f"⚙️ DiT {_dt(DIT_DTYPE)} · text encoder {_dt(TEXT_DTYPE)} · outputs saved to {OUT_DIR}")
print("⏳ Models load on the first Generate click. Logs for each run appear below.")
print("=" * 65 + "\n")

try:  # keep the cell alive so generation logs keep streaming here
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 ScenA Studio stopped.")

---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---